In [9]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [10]:
import zipfile
import os

zip_path = "/content/temp_data.zip"
extract_path = "/content/isl_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [11]:
import os

BASE_PATH = "/content/isl_dataset"

for root, dirs, files in os.walk(BASE_PATH):
    print(root)
    print("Folders:", dirs[:10])
    print("Images:", len([
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ]))
    print("-" * 50)

/content/isl_dataset
Folders: ['data']
Images: 0
--------------------------------------------------
/content/isl_dataset/data
Folders: ['8', 'N', 'Y', 'Z', 'T', '3', '4', 'F', 'J', '5']
Images: 0
--------------------------------------------------
/content/isl_dataset/data/8
Folders: []
Images: 1200
--------------------------------------------------
/content/isl_dataset/data/N
Folders: []
Images: 1200
--------------------------------------------------
/content/isl_dataset/data/Y
Folders: []
Images: 1200
--------------------------------------------------
/content/isl_dataset/data/Z
Folders: []
Images: 1200
--------------------------------------------------
/content/isl_dataset/data/T
Folders: []
Images: 1200
--------------------------------------------------
/content/isl_dataset/data/3
Folders: []
Images: 1200
--------------------------------------------------
/content/isl_dataset/data/4
Folders: []
Images: 1200
--------------------------------------------------
/content/isl_dataset/data

In [12]:
import torch
from torchvision import datasets, transforms
import os

# Dataset path
DATASET_PATH = "/content/isl_dataset/data"

# Basic image transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load dataset
dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)

print("Total images:", len(dataset))
print("Number of classes:", len(dataset.classes))
print("Class names:", dataset.classes)
print("Class mapping:", dataset.class_to_idx)

Total images: 42000
Number of classes: 35
Class names: ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Class mapping: {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, 'A': 9, 'B': 10, 'C': 11, 'D': 12, 'E': 13, 'F': 14, 'G': 15, 'H': 16, 'I': 17, 'J': 18, 'K': 19, 'L': 20, 'M': 21, 'N': 22, 'O': 23, 'P': 24, 'Q': 25, 'R': 26, 'S': 27, 'T': 28, 'U': 29, 'V': 30, 'W': 31, 'X': 32, 'Y': 33, 'Z': 34}


In [13]:
from torch.utils.data import random_split, DataLoader

# 80% training, 20% validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Training images: 33600
Validation images: 8400


In [14]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully!")

DataLoaders created successfully!


In [15]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First 10 labels:", labels[:10])

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
First 10 labels: tensor([ 4,  8, 30, 10, 25,  9,  0, 16, 21, 25])


In [16]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU not detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!nvidia-smi

Fri Sep 18 21:08:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 55.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 67.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 208.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 90.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 152.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [17]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [18]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

# =========================
# 1. Paths
# =========================

DATASET_PATH = "/content/isl_dataset/data"

# =========================
# 2. Image transformations
# =========================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    # Data augmentation
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================
# 3. Load complete dataset
# =========================

full_dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=train_transform
)

print("Total images:", len(full_dataset))
print("Number of classes:", len(full_dataset.classes))
print("Classes:", full_dataset.classes)

Total images: 42000
Number of classes: 35
Classes: ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [19]:
# 80% training
# 20% validation

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Validation dataset should NOT use augmentation
val_dataset.dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=val_transform
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Training images: 33600
Validation images: 8400


In [20]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders ready!")

DataLoaders ready!


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

Device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [22]:
from torchvision import models
from torchvision.models import ResNet18_Weights
import torch.nn as nn

# Load pretrained ResNet18
weights = ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

# Number of classes in our dataset
num_classes = len(full_dataset.classes)

# Replace ResNet's final layer
model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

# Move model to GPU
model = model.to(device)

print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


Linear(in_features=512, out_features=35, bias=True)


In [23]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss function and optimizer ready!")

Loss function and optimizer ready!


In [24]:
NUM_EPOCHS = 15

best_val_accuracy = 0.0

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

MODEL_PATH = "/content/isl_best_model.pth"

for epoch in range(NUM_EPOCHS):

    # =========================
    # TRAIN
    # =========================

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    # =========================
    # PRINT
    # =========================

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )

    # =========================
    # SAVE BEST MODEL
    # =========================

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print(f"  ✓ Best model saved! ({val_accuracy:.2f}%)")

print("\nTraining finished!")
print("Best validation accuracy:", best_val_accuracy)

Epoch [1/15] | Train Loss: 0.0822 | Train Acc: 98.88% | Val Loss: 0.0012 | Val Acc: 100.00%
  ✓ Best model saved! (100.00%)
Epoch [2/15] | Train Loss: 0.0012 | Train Acc: 100.00% | Val Loss: 0.0004 | Val Acc: 100.00%
Epoch [3/15] | Train Loss: 0.0005 | Train Acc: 100.00% | Val Loss: 0.0002 | Val Acc: 100.00%
Epoch [4/15] | Train Loss: 0.0003 | Train Acc: 100.00% | Val Loss: 0.0001 | Val Acc: 100.00%
Epoch [5/15] | Train Loss: 0.0002 | Train Acc: 100.00% | Val Loss: 0.0001 | Val Acc: 100.00%
Epoch [6/15] | Train Loss: 0.0001 | Train Acc: 100.00% | Val Loss: 0.0001 | Val Acc: 100.00%
Epoch [7/15] | Train Loss: 0.0001 | Train Acc: 100.00% | Val Loss: 0.0000 | Val Acc: 100.00%
Epoch [8/15] | Train Loss: 0.0001 | Train Acc: 100.00% | Val Loss: 0.0000 | Val Acc: 100.00%
Epoch [9/15] | Train Loss: 0.0000 | Train Acc: 100.00% | Val Loss: 0.0000 | Val Acc: 100.00%
Epoch [10/15] | Train Loss: 0.0000 | Train Acc: 100.00% | Val Loss: 0.0000 | Val Acc: 100.00%
Epoch [11/15] | Train Loss: 0.0000 | T

In [8]:
import matplotlib.pyplot as plt

epochs = range(1, NUM_EPOCHS + 1)

# =========================
# ACCURACY GRAPH
# =========================

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_accuracies, marker='o', label='Training Accuracy')
plt.plot(epochs, val_accuracies, marker='o', label='Validation Accuracy')

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid()
plt.show()


# =========================
# LOSS GRAPH
# =========================

plt.figure(figsize=(8, 5))

plt.plot(epochs, train_losses, marker='o', label='Training Loss')
plt.plot(epochs, val_losses, marker='o', label='Validation Loss')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid()
plt.show()

NameError: name 'model' is not defined